# Notebook 05 — True Sparse Autoencoder with Top-k Constraint

**Purpose:** replace the Notebook 04 NMF baseline with an actual sparse autoencoder.

Notebook 04 remains the NMF baseline.  
Notebook 05 trains a small neural autoencoder:

```text
X → encoder → top-k sparse latent Z → decoder → X_hat
```

Core claim:

```text
A true sparse autoencoder recovers Mod30 local tile structure
when latent capacity and sparsity are balanced.
```

This notebook keeps the same recovery metrics used in Notebook 04:

```text
coverage
lane purity
reconstruction error
activation sparsity
redundancy
effective_features = captured_lanes / total_components
```


## 0. Bulletproof setup

Run first.

In [ ]:

from pathlib import Path
import sys

def find_repo_root(start=None, marker="src"):
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    return None

REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "mod30-manifold-tiling"
    SRC_DIR = REPO_ROOT / "src"
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    (SRC_DIR / "__init__.py").write_text("", encoding="utf-8")
    (SRC_DIR / "mod30.py").write_text("""from math import gcd
MOD30 = 30
MOD30_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
def mod_index(n, mod): return n % mod
def mod_mask(n, residues, mod): return mod_index(n, mod) in residues
def generate_coprime_residues(mod): return [r for r in range(1, mod) if gcd(r, mod) == 1]
def mod30_index(n): return mod_index(n, MOD30)
def mod30_mask(n): return mod_mask(n, MOD30_RESIDUES, MOD30)
def residue_to_lane_index(residue): return MOD30_RESIDUES.index(residue) if residue in MOD30_RESIDUES else -1
def mod30_residues(n_max): return [n for n in range(2, n_max) if mod30_mask(n)]
""", encoding="utf-8")
    (SRC_DIR / "tiling_metrics.py").write_text("""import numpy as np
def lane_coverage(captured_residues, target_residues):
    captured, target = set(captured_residues), set(target_residues)
    hit, missed = captured & target, target - captured
    return {"target_lanes": len(target), "captured_lanes": len(hit), "missed_lanes": len(missed),
            "coverage_fraction": len(hit)/len(target) if target else 0.0,
            "captured_residues": sorted(hit), "missed_residues": sorted(missed)}
def reconstruction_error(X, X_hat):
    denom = np.linalg.norm(X)
    return 0.0 if denom == 0 else float(np.linalg.norm(X - X_hat) / denom)
def activation_sparsity(A, eps=1e-6): return float(np.mean(A <= eps))
def feature_lane_alignment(A, lane_labels, residues):
    align = np.zeros((A.shape[1], len(residues)))
    for j in range(A.shape[1]):
        total = A[:, j].sum()
        if total <= 0: continue
        for k, r in enumerate(residues):
            mask = lane_labels == r
            align[j, k] = A[mask, j].sum() / total
    return align
def lane_purity_from_alignment(align): return [] if align.size == 0 else align.max(axis=1).tolist()
def recovered_residues_from_alignment(align, residues, threshold=0.45):
    recovered = []
    for row in align:
        if row.max() >= threshold:
            recovered.append(residues[int(row.argmax())])
    return sorted(set(recovered))
""", encoding="utf-8")
    (SRC_DIR / "plots.py").write_text("""import matplotlib.pyplot as plt
def save_current(path, dpi=180):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    return path
""", encoding="utf-8")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "figures"
DATA_DIR = REPO_ROOT / "data"
OUTPUTS_DIR = REPO_ROOT / "outputs"
for d in [FIGURES_DIR, DATA_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("src exists:", (REPO_ROOT / "src").exists())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise ImportError("This notebook needs PyTorch. In Colab, Runtime usually includes torch by default.") from e

from src.mod30 import MOD30_RESIDUES, mod30_index, mod30_mask, residue_to_lane_index
from src.tiling_metrics import (
    lane_coverage,
    reconstruction_error,
    activation_sparsity,
    feature_lane_alignment,
    lane_purity_from_alignment,
    recovered_residues_from_alignment,
)
from src.plots import save_current

SEED = 9423
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("Persisting Mod30 lanes:", MOD30_RESIDUES)


## 1. Build Mod30 dataset

Same synthetic setup as Notebook 04 so NMF and SAE results remain comparable.


In [ ]:
n_min = 1
n_max = 1500
values = np.arange(n_min, n_max + 1)

df = pd.DataFrame({
    "n": values,
    "mod30_residue": [mod30_index(int(n)) for n in values],
})
df["inside_mod30_gate"] = df["n"].apply(lambda n: mod30_mask(int(n)))
df["lane_index"] = df["mod30_residue"].apply(residue_to_lane_index)

gated_df = df[df["inside_mod30_gate"]].copy().reset_index(drop=True)
gated_df.head()


## 2. Continuous synthetic embeddings

Each gated integer receives a continuous vector:

```text
circle coordinates + lane channels + small noise
```


In [ ]:
RNG = np.random.default_rng(SEED)

def build_continuous_embedding(gated_df, noise_scale=0.04, lane_strength=1.0):
    residues = gated_df["mod30_residue"].to_numpy()
    lane_idx = gated_df["lane_index"].to_numpy()

    theta = 2 * np.pi * residues / 30.0

    circle = np.column_stack([
        0.5 + 0.5 * np.cos(theta),
        0.5 + 0.5 * np.sin(theta),
    ])

    lane_channels = np.zeros((len(gated_df), len(MOD30_RESIDUES)))
    lane_channels[np.arange(len(gated_df)), lane_idx] = lane_strength

    # local overlap: makes recovery nontrivial and SAE-like
    for i in range(len(gated_df)):
        k = lane_idx[i]
        lane_channels[i, (k - 1) % len(MOD30_RESIDUES)] += 0.18
        lane_channels[i, (k + 1) % len(MOD30_RESIDUES)] += 0.18

    noise = np.abs(RNG.normal(loc=0.0, scale=noise_scale, size=(len(gated_df), 4)))

    X = np.column_stack([circle, lane_channels, noise])
    X = MinMaxScaler().fit_transform(X)
    return X.astype(np.float32)

X = build_continuous_embedding(gated_df)
X_tensor = torch.tensor(X, dtype=torch.float32, device=device)
X.shape


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=gated_df["lane_index"], s=10)
plt.xlabel("embedding dim 0")
plt.ylabel("embedding dim 1")
plt.title("Synthetic continuous embedding for SAE training")
save_current(FIGURES_DIR / "23_sae_synthetic_embedding.png")
plt.show()


## 3. Define top-k sparse autoencoder

The key operation is `topk_mask`, which keeps only the largest latent activations per sample.

This mimics a k-sparse SAE:

```text
dense latent → top-k sparse latent
```


In [ ]:
def topk_mask(z, k):
    if k >= z.shape[1]:
        return z
    values, indices = torch.topk(z, k=k, dim=1)
    mask = torch.zeros_like(z)
    mask.scatter_(1, indices, 1.0)
    return z * mask

class TopKSparseAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim, top_k):
        super().__init__()
        self.encoder = nn.Linear(input_dim, latent_dim)
        self.decoder = nn.Linear(latent_dim, input_dim, bias=True)
        self.top_k = top_k

    def forward(self, x):
        z_dense = F.relu(self.encoder(x))
        z_sparse = topk_mask(z_dense, self.top_k)
        x_hat = self.decoder(z_sparse)
        return x_hat, z_sparse, z_dense


## 4. Train one sparse autoencoder

Training objective:

```text
reconstruction MSE + small L1 penalty on dense latent
```

Top-k enforces hard sparsity; L1 encourages cleaner latent magnitude.


In [ ]:
def train_sae(latent_dim=8, top_k=1, epochs=1200, lr=1e-2, l1_weight=1e-4, verbose=False):
    torch.manual_seed(SEED + latent_dim * 100 + top_k)
    model = TopKSparseAutoencoder(X.shape[1], latent_dim, top_k).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []

    for epoch in range(epochs):
        optimizer.zero_grad()
        x_hat, z_sparse, z_dense = model(X_tensor)

        recon_loss = F.mse_loss(x_hat, X_tensor)
        l1_loss = z_dense.abs().mean()
        loss = recon_loss + l1_weight * l1_loss

        loss.backward()
        optimizer.step()

        losses.append(float(loss.detach().cpu()))

        if verbose and epoch % 200 == 0:
            print(epoch, losses[-1])

    with torch.no_grad():
        x_hat, z_sparse, z_dense = model(X_tensor)

    return {
        "model": model,
        "losses": losses,
        "X_hat": x_hat.detach().cpu().numpy(),
        "Z": z_sparse.detach().cpu().numpy(),
        "Z_dense": z_dense.detach().cpu().numpy(),
    }

demo = train_sae(latent_dim=8, top_k=1, epochs=800, lr=1e-2)
demo["Z"].shape


## 5. Evaluate SAE recovery

Metrics match Notebook 04, with one addition:

```text
effective_features = captured_lanes / total_components
```

Interpretation:

```text
1.0 = all components useful
<1.0 = unused or redundant capacity
```


In [ ]:
def evaluate_sae_result(result, latent_dim, top_k, threshold=0.35):
    Z = result["Z"]
    X_hat = result["X_hat"]

    align = feature_lane_alignment(
        Z,
        gated_df["mod30_residue"].to_numpy(),
        MOD30_RESIDUES,
    )
    purities = lane_purity_from_alignment(align)
    recovered = recovered_residues_from_alignment(align, MOD30_RESIDUES, threshold=threshold)
    cov = lane_coverage(recovered, MOD30_RESIDUES)

    total_activations = float((Z > 1e-6).sum())
    redundancy = total_activations / cov["captured_lanes"] if cov["captured_lanes"] else np.inf
    effective_features = cov["captured_lanes"] / latent_dim if latent_dim else 0.0

    return {
        "latent_dim": latent_dim,
        "top_k": top_k,
        "reconstruction_error": reconstruction_error(X, X_hat),
        "activation_sparsity": activation_sparsity(Z, eps=1e-6),
        "mean_lane_purity": float(np.mean(purities)) if purities else 0.0,
        "max_lane_purity": float(np.max(purities)) if purities else 0.0,
        "captured_lanes": cov["captured_lanes"],
        "coverage_fraction": cov["coverage_fraction"],
        "total_activations": total_activations,
        "redundancy": redundancy,
        "effective_features": effective_features,
        "recovered_residues": recovered,
        "missed_residues": cov["missed_residues"],
        "alignment": align,
    }

demo_metrics = evaluate_sae_result(demo, latent_dim=8, top_k=1)
{k: v for k, v in demo_metrics.items() if k != "alignment"}


## 6. Train SAE grid

Test a small capacity/sparsity grid:

```text
latent_dim = 1, 4, 8, 12
top_k = 1 or 2
```

This mirrors the Notebook 04 component sweep while adding explicit sparsity control.


In [ ]:
grid = [
    {"latent_dim": 1, "top_k": 1},
    {"latent_dim": 4, "top_k": 1},
    {"latent_dim": 8, "top_k": 1},
    {"latent_dim": 8, "top_k": 2},
    {"latent_dim": 12, "top_k": 1},
    {"latent_dim": 12, "top_k": 2},
]

sae_results = {}
sae_metrics_rows = []

for cfg in grid:
    latent_dim = cfg["latent_dim"]
    top_k = cfg["top_k"]
    result = train_sae(latent_dim=latent_dim, top_k=top_k, epochs=1200, lr=1e-2, l1_weight=1e-4)
    metric = evaluate_sae_result(result, latent_dim=latent_dim, top_k=top_k, threshold=0.35)

    key = (latent_dim, top_k)
    sae_results[key] = {**result, "metrics": metric}
    row = {k: v for k, v in metric.items() if k != "alignment"}
    sae_metrics_rows.append(row)

sae_metrics_df = pd.DataFrame(sae_metrics_rows)
sae_metrics_df.to_csv(DATA_DIR / "05_sae_recovery_metrics.csv", index=False)
sae_metrics_df


## 7. SAE learned activation matrix

Show the matched-capacity case:

```text
latent_dim = 8, top_k = 1
```


In [ ]:
show_key = (8, 1)
Z_show = sae_results[show_key]["Z"]

plt.figure(figsize=(12, 4.8))
plt.imshow(Z_show[:240].T, aspect="auto", interpolation="nearest")
plt.xlabel("sample index")
plt.ylabel("SAE latent")
plt.title("SAE learned activation matrix: latent_dim=8, top_k=1")
save_current(FIGURES_DIR / "24_sae_activation_matrix_8_topk1.png")
plt.show()


## 8. SAE feature-to-lane alignment

Rows are learned latent features.  
Columns are true Mod30 residue lanes.


In [ ]:
align_show = sae_results[show_key]["metrics"]["alignment"]

plt.figure(figsize=(8, 5))
plt.imshow(align_show, aspect="auto", interpolation="nearest")
plt.xticks(range(len(MOD30_RESIDUES)), [f"r{r}" for r in MOD30_RESIDUES])
plt.yticks(range(show_key[0]), [f"Z{j}" for j in range(show_key[0])])
plt.xlabel("true residue lane")
plt.ylabel("SAE latent")
plt.title("SAE feature-to-lane alignment: latent_dim=8, top_k=1")
plt.colorbar(label="fraction of latent activation")
save_current(FIGURES_DIR / "25_sae_feature_to_lane_alignment_8_topk1.png")
plt.show()


## 9. Metrics summary

This plot includes the new metric:

```text
effective_features = captured_lanes / total_components
```


In [ ]:
plot_df = sae_metrics_df.copy()
plot_df["setting"] = plot_df.apply(lambda r: f"L{int(r.latent_dim)}-k{int(r.top_k)}", axis=1)

x = np.arange(len(plot_df))
width = 0.14

plt.figure(figsize=(12, 5.8))
plt.bar(x - 2*width, plot_df["coverage_fraction"], width, label="coverage")
plt.bar(x - width, plot_df["mean_lane_purity"], width, label="mean purity")
plt.bar(x, 1 - plot_df["reconstruction_error"], width, label="1 - recon error")
plt.bar(x + width, plot_df["activation_sparsity"], width, label="activation sparsity")
plt.bar(x + 2*width, plot_df["effective_features"], width, label="effective features")

plt.xticks(x, plot_df["setting"], rotation=20)
plt.xlabel("SAE setting")
plt.ylabel("metric value")
plt.title("SAE recovery metrics: capacity, sparsity, effective features")
plt.legend()
save_current(FIGURES_DIR / "26_sae_recovery_metrics_summary.png")
plt.show()


## 10. Reconstruction and redundancy tradeoff

This plot tracks:

```text
reconstruction error
redundancy
effective_features
```


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(plot_df["redundancy"], plot_df["coverage_fraction"], s=120)

for _, row in plot_df.iterrows():
    plt.annotate(row["setting"], (row["redundancy"], row["coverage_fraction"]),
                 textcoords="offset points", xytext=(6, 6), ha="left")

plt.xlabel("redundancy = total activations / captured lanes")
plt.ylabel("coverage fraction")
plt.title("SAE redundancy vs coverage")
save_current(FIGURES_DIR / "27_sae_redundancy_vs_coverage.png")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(plot_df["effective_features"], 1 - plot_df["reconstruction_error"], s=120)

for _, row in plot_df.iterrows():
    plt.annotate(row["setting"], (row["effective_features"], 1 - row["reconstruction_error"]),
                 textcoords="offset points", xytext=(6, 6), ha="left")

plt.xlabel("effective_features = captured_lanes / latent_dim")
plt.ylabel("1 - reconstruction error")
plt.title("SAE useful capacity vs reconstruction")
save_current(FIGURES_DIR / "28_sae_effective_features_vs_reconstruction.png")
plt.show()


## 11. Alignment matrices across SAE settings

These plots show undercomplete, matched, and overcomplete behavior.


In [ ]:
for key, result in sae_results.items():
    latent_dim, top_k = key
    align = result["metrics"]["alignment"]

    plt.figure(figsize=(8, max(2.5, 0.42 * latent_dim)))
    plt.imshow(align, aspect="auto", interpolation="nearest")
    plt.xticks(range(len(MOD30_RESIDUES)), [f"r{r}" for r in MOD30_RESIDUES])
    plt.yticks(range(latent_dim), [f"Z{j}" for j in range(latent_dim)])
    plt.xlabel("true residue lane")
    plt.ylabel("SAE latent")
    plt.title(f"SAE feature-to-lane alignment: latent_dim={latent_dim}, top_k={top_k}")
    plt.colorbar(label="fraction of latent activation")
    save_current(FIGURES_DIR / f"29_sae_alignment_L{latent_dim}_k{top_k}.png")
    plt.show()


## 12. Training loss curves

Loss curves confirm optimization behavior for each setting.


In [ ]:
plt.figure(figsize=(9, 5))
for key, result in sae_results.items():
    latent_dim, top_k = key
    plt.plot(result["losses"], label=f"L{latent_dim}-k{top_k}", alpha=0.85)

plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("SAE training loss curves")
plt.legend()
save_current(FIGURES_DIR / "30_sae_training_loss_curves.png")
plt.show()


## 13. Interpretation

Expected regimes:

| Setting | Interpretation |
|---|---|
| L1-k1 | global compression / superposition |
| L4-k1 | grouped sparse tiles |
| L8-k1 | matched sparse local tile recovery |
| L8-k2 | more overlap, lower sparsity |
| L12-k1 | overcomplete sparse recovery |
| L12-k2 | overcomplete + redundant overlap |

Paper-facing sentence:

```text
The true sparse autoencoder reproduces the same regimes as the NMF baseline:
undercomplete global blur, matched local tile recovery, and overcomplete redundancy.
```


## 14. Save compact summary

In [ ]:
summary_md = f"""# Notebook 05 Summary — True Sparse Autoencoder

Notebook 05 replaces the Notebook 04 NMF baseline with a top-k sparse autoencoder.

## Model

`X → encoder → top-k sparse latent Z → decoder → X_hat`

## Metrics

{sae_metrics_df.to_markdown(index=False)}

## Key metric

`effective_features = captured_lanes / total_components`

## Interpretation

The true sparse autoencoder reproduces the same regimes as the NMF baseline:

1. undercomplete global blur / superposition
2. grouped sparse tiles
3. matched local Mod30 tile recovery
4. overcomplete redundancy

## Generated figures

- `figures/23_sae_synthetic_embedding.png`
- `figures/24_sae_activation_matrix_8_topk1.png`
- `figures/25_sae_feature_to_lane_alignment_8_topk1.png`
- `figures/26_sae_recovery_metrics_summary.png`
- `figures/27_sae_redundancy_vs_coverage.png`
- `figures/28_sae_effective_features_vs_reconstruction.png`
- `figures/29_sae_alignment_L*_k*.png`
- `figures/30_sae_training_loss_curves.png`

## Generated data

- `data/05_sae_recovery_metrics.csv`
"""

summary_path = OUTPUTS_DIR / "05_true_sparse_autoencoder_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
print(summary_path)


## 15. Optional: zip-download pattern

Uncomment in Colab to download figures, data, and output summaries.


In [ ]:
# Optional zip-download pattern:
#
# import shutil
#
# bundle_name = "notebook_05_true_sparse_autoencoder_outputs"
# bundle_base = REPO_ROOT / bundle_name
# bundle_zip = REPO_ROOT / f"{bundle_name}.zip"
#
# if bundle_base.exists():
#     shutil.rmtree(bundle_base)
#
# bundle_base.mkdir(parents=True, exist_ok=True)
#
# for folder_name in ["figures", "data", "outputs"]:
#     src_folder = REPO_ROOT / folder_name
#     dst_folder = bundle_base / folder_name
#     if src_folder.exists():
#         shutil.copytree(src_folder, dst_folder)
#
# shutil.make_archive(str(bundle_base), "zip", bundle_base)
# print("Created:", bundle_zip)
#
# # In Google Colab, uncomment:
# # from google.colab import files
# # files.download(str(bundle_zip))


## 16. Recommended next step

The repo now has enough notebook evidence for a first paper draft.

Recommended file:

```text
paper/mod30_manifold_tiling.tex
```

Working title:

```text
Residue-Class Tiling as a Finite Analogue for Sparse Feature Capture
```
